<a href="https://colab.research.google.com/github/pronabpaul/Q1_Effective_Rank_Predicts_Synthetic_Corruption/blob/main/Q1_Effective_Rank_vs_Synthetic_Corruptions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Q1. Does effective rank predict performance drop under synthetic corruptions?**

**First report in the series:** *Source Representation Geometry Predicts Failure Under Domain Shift in Medical Imaging*  
**Part 1 of 10**



## 1. Introduction

Medical AI models often fail when deployed across different hospitals, scanners, or imaging modalities, a challenge known as domain shift. Existing methods to detect or mitigate such failures typically require access to target domain data (labelled or unlabelled) during development or at test time. In real-world clinical settings, however, target data may be unavailable, privacy-restricted, or costly to acquire. As a result, there is currently no practical way to assess, **before deployment**, whether a model will fail under an unseen domain shift.

This study investigates whether **geometric properties of a model's representations**, measured **only on source data**, can predict performance degradation under domain shift without any target access. Such a predictor could serve as a prospective risk score, enabling safer deployment decisions in clinical practice.

The report presents the first experiment (Q1) in a series of 10 investigations. The full series addresses effective rank, spectral decay, subspace alignment, architecture generality, layer-wise predictive power, stability, and metric fusion. The experiment tests whether **effective rank** predicts performance drop under **synthetic corruptions** (noise, blur, contrast).



## 2. Hypothesis and Decision Rule

**Question:** Does a lower effective rank of source features correlate with a larger performance drop under synthetic corruptions (noise, blur, contrast)?

**H₁:** The Pearson correlation ρ between effective rank and average performance drop is **less than -0.3** (negative correlation; lower effective rank => larger drop).

**Decision rule for H₁ supported:**  
The 95% bootstrap confidence interval (CI) for ρ lies **entirely below -0.2** (i.e., the upper bound of the CI is less than -0.2).  
If this condition is not met, H₁ is rejected.



## 3. Methods

| Aspect | Details |
|--------|---------|
| **Dataset** | CheXpert (5000 images, fixed random subset with `random_state=42`) |
| **Task** | Binary classification of Cardiomegaly |
| **Model** | ResNet-18 (pretrained on ImageNet) |
| **Training** | 10 epochs, Adam (`lr=1e-4`), batch size 64 |
| **Regularisation** | Weight decay (`1e-5` to `1e-2`) and dropout (`0` to `0.5`) randomly sampled per model |
| **Corruptions** | Noise (σ=0.1, 0.2, 0.3), Blur (σ=0.5, 1.0, 2.0), Contrast (factor 0.5, 1.5, 2.0) - 9 conditions total |
| **Drop metric** | Average over all conditions: `drop = AUC(clean) - AUC(corrupted)` |
| **Effective rank** | SVD of centred feature covariance on validation features (penultimate layer) |
| **Statistical analysis** | Pearson r, Spearman r, 95% bootstrap CI (1000 resamples) over 100 independent models |




In [ ]:
# Q1: Effective rank predicts performance drop under synthetic corruptions

!pip install -q kagglehub pandas scikit-learn matplotlib tqdm kornia

import os, random, warnings
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
from pathlib import Path
import kornia.filters as K
warnings.filterwarnings('ignore')

BATCH_SIZE = 64
EPOCHS = 10
N_MODELS = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {DEVICE}")

import kagglehub
dataset_path = kagglehub.dataset_download("ashery/chexpert")
dataset_path = Path(dataset_path)

data_root = None
for sub in dataset_path.iterdir():
    if sub.is_dir() and (sub / 'train.csv').exists():
        data_root = sub
        break
if data_root is None and (dataset_path / 'train.csv').exists():
    data_root = dataset_path
if data_root is None:
    raise RuntimeError("train.csv not found")

TRAIN_CSV = data_root / 'train.csv'
IMG_DIR = data_root

df = pd.read_csv(TRAIN_CSV)
df['Cardiomegaly'] = df['Cardiomegaly'].fillna(0).replace(-1, 0)

first_path = df.iloc[0]['Path']
if first_path.startswith('CheXpert-v1.0-small/'):
    prefix = 'CheXpert-v1.0-small/'
    print(f"Stripping prefix '{prefix}'")
else:
    prefix = None

# Sample a subset of rows for efficiency
sample_rows = df.sample(n=6000, random_state=42)

samples = []
for _, row in tqdm(sample_rows.iterrows(), total=len(sample_rows), desc="Checking images"):
    rel_path = row['Path']
    if prefix:
        rel_path = rel_path.replace(prefix, '', 1)
    full_path = IMG_DIR / rel_path
    if full_path.exists():
        samples.append((str(full_path), row['Cardiomegaly']))
    if len(samples) >= 5000:
        break

print(f"Found {len(samples)} images")
if len(samples) == 0:
    raise RuntimeError("No images found")

random.seed(42)
random.shuffle(samples)
split = int(0.8 * len(samples))
train_samples = samples[:split]
val_samples = samples[split:]
print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")

class CheXpertDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)

train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

train_ds = CheXpertDataset(train_samples, train_tf)
val_ds   = CheXpertDataset(val_samples, val_tf)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2)

def effective_rank(features):
    centered = features - features.mean(0, keepdims=True)
    s = np.linalg.svd(centered, full_matrices=False)[1]
    probs = s**2 / (np.sum(s**2) + 1e-12)
    entropy = -np.sum(probs * np.log(probs + 1e-12))
    return np.exp(entropy)

def apply_corruption_batch(images, corr_type, level):
    mean = torch.tensor([0.485,0.456,0.406], device=images.device).view(1,3,1,1)
    std  = torch.tensor([0.229,0.224,0.225], device=images.device).view(1,3,1,1)
    img = images * std + mean
    img = torch.clamp(img, 0, 1)

    if corr_type == 'noise':
        corrupted = img + torch.randn_like(img) * level
    elif corr_type == 'blur':
        k = int(level * 10) // 2 * 2 + 1
        sigma = float(level * 2.0)
        corrupted = K.gaussian_blur2d(img, (k, k), (sigma, sigma))
    elif corr_type == 'contrast':
        corrupted = img * level
    else:
        corrupted = img
    corrupted = torch.clamp(corrupted, 0, 1)
    return (corrupted - mean) / std

def train_and_evaluate(seed, weight_decay, dropout):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(512,1))
    model.to(DEVICE)
    crit = nn.BCEWithLogitsLoss()
    opt = optim.Adam(model.parameters(), lr=1e-4, weight_decay=weight_decay)

    for _ in range(EPOCHS):
        model.train()
        for imgs, labs in train_loader:
            imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(imgs).squeeze(), labs)
            loss.backward()
            opt.step()

    model.eval()
    # Effective rank on validation features
    all_feats = []
    with torch.no_grad():
        for imgs, _ in val_loader:
            imgs = imgs.to(DEVICE)
            x = model.conv1(imgs); x = model.bn1(x); x = model.relu(x)
            x = model.maxpool(x); x = model.layer1(x); x = model.layer2(x)
            x = model.layer3(x); x = model.layer4(x); x = model.avgpool(x)
            all_feats.append(torch.flatten(x, 1).cpu().numpy())
    val_feats = np.concatenate(all_feats, axis=0)
    eff = effective_rank(val_feats)

    # Source AUC
    src_probs, src_labs = [], []
    with torch.no_grad():
        for imgs, labs in val_loader:
            imgs = imgs.to(DEVICE)
            probs = torch.sigmoid(model(imgs).squeeze()).cpu().numpy()
            src_probs.extend(probs); src_labs.extend(labs.numpy())
    src_auc = roc_auc_score(src_labs, src_probs) if len(np.unique(src_labs))>1 else 0.5

    # Corruption evaluation
    corruptions = {'noise':[0.1,0.2,0.3], 'blur':[0.5,1.0,2.0], 'contrast':[0.5,1.5,2.0]}
    drops = []
    with torch.no_grad():
        for ctype, levels in corruptions.items():
            for lvl in levels:
                all_probs, all_labs = [], []
                for imgs, labs in val_loader:
                    imgs = imgs.to(DEVICE)
                    corrupted = apply_corruption_batch(imgs, ctype, lvl)
                    logits = model(corrupted).squeeze()
                    probs = torch.sigmoid(logits).detach().cpu().numpy()  # FIXED: added .detach()
                    all_probs.extend(probs); all_labs.extend(labs.numpy())
                tgt_auc = roc_auc_score(all_labs, all_probs) if len(np.unique(all_labs))>1 else 0.5
                drops.append(src_auc - tgt_auc)
    return eff, np.mean(drops)

results = []
for i in range(N_MODELS):
    seed = i * 777 + 42
    wd = 10 ** np.random.uniform(-5, -2)
    dr = np.random.uniform(0, 0.5)
    print(f"Model {i+1}/{N_MODELS} | wd={wd:.2e} dropout={dr:.2f}", end=' ')
    eff, drop = train_and_evaluate(seed, wd, dr)
    results.append({'eff_rank': eff, 'drop': drop})
    print(f"rank={eff:.2f} drop={drop:.4f}")

df = pd.DataFrame(results)

r_pear, p_pear = pearsonr(df['eff_rank'], df['drop'])
r_spear, p_spear = spearmanr(df['eff_rank'], df['drop'])
print(f"\nPearson r = {r_pear:.3f} (p={p_pear:.4f})")
print(f"Spearman r = {r_spear:.3f} (p={p_spear:.4f})")

boot = []
for _ in range(1000):
    boot_df = df.sample(n=len(df), replace=True)
    r_b, _ = pearsonr(boot_df['eff_rank'], boot_df['drop'])
    boot.append(r_b)
ci_low, ci_high = np.percentile(boot, [2.5,97.5])
print(f"95% bootstrap CI: [{ci_low:.3f}, {ci_high:.3f}]")

if ci_high < -0.2:
    print("\nH1 supported: Upper bound of 95% CI < -0.2")
else:
    print("\nH1 not supported: CI does not lie entirely below -0.2")

plt.scatter(df['eff_rank'], df['drop'], alpha=0.6)
plt.xlabel('Effective Rank')
plt.ylabel('Average Performance Drop')
plt.title('Effective Rank vs Synthetic Corruption Drop')
plt.grid(True)
plt.show()

## 4. Results

### 4.1 Summary Statistics (n = 131)

| Statistic | Effective Rank | Average Drop |
|-----------|----------------|--------------|
| **Mean** | 61.46 | 0.1127 |
| **Std. Dev.** | 24.56 | 0.0223 |
| **Minimum** | 12.55 | 0.0550 |
| **Maximum** | 106.05 | 0.1696 |
| **Range** | 93.50 | 0.1146 |

The models exhibit a wide spread in effective rank (12.55 to 106.05), essential for a meaningful correlation analysis. The average drop ranges from 0.055 to 0.170, indicating that all models suffer some performance degradation under corruptions.

**Note:** The 131 models were pooled from Colab and Kaggle runs.

### 4.2 Correlation Analysis (n = 131)

| Correlation | Value | p-value |
|-------------|-------|---------|
| **Pearson r** | **0.198** | **0.023** |
| **Spearman r** | **0.214** | **0.014** |
| **95% bootstrap CI (Pearson)** | **[0.025, 0.368]** | – |

Both Pearson and Spearman correlations are **positive and statistically significant** (p < 0.05). The 95% bootstrap CI lies entirely above zero, indicating a weak but consistent positive relationship.

### 4.3 Hyperparameter Ranges (n = 131)

| Hyperparameter | Min | Max | Mean |
|----------------|-----|-----|------|
| **Weight decay** | 1.02x10⁻⁵ | 9.78x10⁻³ | 1.48x10⁻³ |
| **Dropout** | 0.00 | 0.49 | 0.25 |

The hyperparameters span broad ranges covering typical values used in practice, ensuring that the observed correlation is not an artefact of a narrow sweep.

## 5. Decision on H₁

**H₁ is NOT supported.**

The 95% bootstrap CI for Pearson r is **[0.025, 0.368]**, which does **not** satisfy the condition `upper bound < -0.2`. The entire interval lies above zero, indicating a weak but significant **positive** correlation - the opposite of the hypothesised direction.

## 6. Interpretation

- The correlation is **weakly positive**: models with higher effective rank tend to have *larger* performance drops under synthetic corruptions.
- The result is **statistically significant** (p < 0.05) but the effect size is small (r ≈ 0.2), meaning effective rank alone explains only about 4% of the variance in performance drop.
- The direction is contrary to the hypothesis that collapsed representations (low rank) would be more fragile. Higher-rank representations may be more sensitive to input perturbations as they encode more features that can be disrupted. Alternatively, the relationship may be nonlinear or influenced by factors not captured by a simple linear correlation.


## 7. Conclusion

The hypothesis that lower effective rank predicts larger performance drops under synthetic corruptions is **not supported**. The correlation was weakly positive (r = 0.198, 95% CI [0.025, 0.368]), opposite to the predicted direction, and the confidence interval does not satisfy the pre-registered decision rule (upper bound < -0.2). **H₁ is rejected.**

Effective rank alone explains only about 4% of the variance in corruption robustness, suggesting it is not a reliable predictor for this setting. The weak positive trend may indicate that higher-rank representations are more sensitive to input perturbations, or that the relationship is nonlinear and requires more refined analysis.

These findings underscore the need to explore other geometry metrics (spectral decay, subspace alignment) and more realistic domain shifts in the subsequent reports of this series.



## 8. Appendix: Raw Data (131 Models)

| # | Rank | Drop | # | Rank | Drop | # | Rank | Drop |
|---|------|------|---|------|------|---|------|------|
| 1 | 83.50 | 0.1091 | 45 | 50.41 | 0.1134 | 89 | 57.63 | 0.1043 |
| 2 | 40.43 | 0.0733 | 46 | 78.26 | 0.1113 | 90 | 74.24 | 0.1315 |
| 3 | 62.63 | 0.0550 | 47 | 65.42 | 0.1312 | 91 | 73.42 | 0.1492 |
| 4 | 42.32 | 0.0860 | 48 | 89.47 | 0.1549 | 92 | 55.99 | 0.1303 |
| 5 | 35.31 | 0.0812 | 49 | 60.18 | 0.1318 | 93 | 85.60 | 0.1018 |
| 6 | 38.05 | 0.1264 | 50 | 36.08 | 0.0999 | 94 | 55.46 | 0.1253 |
| 7 | 88.87 | 0.1148 | 51 | 91.32 | 0.1053 | 95 | 12.55 | 0.1300 |
| 8 | 82.82 | 0.1157 | 52 | 88.48 | 0.1260 | 96 | 74.87 | 0.1272 |
| 9 | 83.57 | 0.1240 | 53 | 42.50 | 0.1008 | 97 | 89.51 | 0.0941 |
| 10 | 64.11 | 0.1212 | 54 | 54.50 | 0.1696 | 98 | 59.73 | 0.1007 |
| 11 | 69.08 | 0.0743 | 55 | 25.24 | 0.1343 | 99 | 82.27 | 0.1294 |
| 12 | 88.96 | 0.1418 | 56 | 25.38 | 0.1180 | 100 | 54.71 | 0.1054 |
| 13 | 44.18 | 0.1401 | 57 | 71.67 | 0.0946 | 101 | 66.53 | 0.1103 |
| 14 | 50.22 | 0.1214 | 58 | 87.66 | 0.1581 | 102 | 72.60 | 0.1063 |
| 15 | 93.46 | 0.1065 | 59 | 106.05 | 0.1052 | 103 | 75.80 | 0.1154 |
| 16 | 63.93 | 0.1435 | 60 | 87.40 | 0.1426 | 104 | 67.17 | 0.1339 |
| 17 | 69.51 | 0.1361 | 61 | 84.93 | 0.1023 | 105 | 42.74 | 0.0763 |
| 18 | 61.53 | 0.1169 | 62 | 88.77 | 0.1212 | 106 | 79.70 | 0.1228 |
| 19 | 33.24 | 0.1076 | 63 | 65.66 | 0.1189 | 107 | 64.04 | 0.0775 |
| 20 | 91.17 | 0.1375 | 64 | 58.58 | 0.1387 | 108 | 81.12 | 0.1010 |
| 21 | 97.28 | 0.1373 | 65 | 35.47 | 0.1151 | 109 | 93.43 | 0.1419 |
| 22 | 26.40 | 0.0980 | 66 | 31.45 | 0.0828 | 110 | 17.95 | 0.1164 |
| 23 | 58.84 | 0.1586 | 67 | 60.88 | 0.1365 | 111 | 57.28 | 0.1089 |
| 24 | 15.05 | 0.1224 | 68 | 71.65 | 0.0690 | 112 | 61.84 | 0.1067 |
| 25 | 54.56 | 0.1280 | 69 | 48.46 | 0.0786 | 113 | 73.18 | 0.1102 |
| 26 | 36.69 | 0.0922 | 70 | 86.76 | 0.1535 | 114 | 76.13 | 0.1345 |
| 27 | 65.44 | 0.0856 | 71 | 72.81 | 0.1152 | 115 | 76.38 | 0.1394 |
| 28 | 31.35 | 0.0908 | 72 | 32.13 | 0.1122 | 116 | 49.93 | 0.1105 |
| 29 | 30.36 | 0.0944 | 73 | 93.73 | 0.1275 | 117 | 92.17 | 0.1116 |
| 30 | 34.18 | 0.1331 | 74 | 66.03 | 0.1160 | 118 | 48.14 | 0.1376 |
| 31 | 97.30 | 0.1111 | 75 | 51.36 | 0.0954 | 119 | 79.74 | 0.1251 |
| 32 | 63.39 | 0.1115 | 76 | 82.51 | 0.1419 | 120 | 62.44 | 0.0798 |
| 33 | 46.64 | 0.0772 | 77 | 28.97 | 0.1102 | 121 | 54.48 | 0.1061 |
| 34 | 65.22 | 0.0913 | 78 | 26.96 | 0.1020 | 122 | 83.53 | 0.1570 |
| 35 | 30.28 | 0.0928 | 79 | 76.51 | 0.0969 | 123 | 72.92 | 0.0936 |
| 36 | 31.46 | 0.1116 | 80 | 18.26 | 0.0803 | 124 | 86.50 | 0.1228 |
| 37 | 42.52 | 0.1072 | 81 | 19.76 | 0.0870 | 125 | 34.71 | 0.0956 |
| 38 | 91.74 | 0.1323 | 82 | 70.11 | 0.1135 | 126 | 82.62 | 0.1219 |
| 39 | 84.33 | 0.0911 | 83 | 37.14 | 0.1100 | 127 | 24.51 | 0.1422 |
| 40 | 91.38 | 0.0765 | 84 | 87.06 | 0.1407 | 128 | 33.57 | 0.0860 |
| 41 | 61.75 | 0.1188 | 85 | 55.68 | 0.1043 | 129 | 86.29 | 0.1082 |
| 42 | 66.63 | 0.1174 | 86 | 70.97 | 0.1063 | 130 | 69.75 | 0.1102 |
| 43 | 74.71 | 0.1069 | 87 | 97.30 | 0.0610 | 131 | 28.45 | 0.1058 |
| 44 | 53.00 | 0.1248 | 88 | 76.79 | 0.1269 | - | - | - |